<a href="https://colab.research.google.com/github/vidyawadhai13/OS-project/blob/main/TAE_1(OS).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import copy

MAX_DEPT = 5
MAX_RES = 2  # printers, scanners

# Global variables
Total = [0] * MAX_RES
Alloc = [[0] * MAX_RES for _ in range(MAX_DEPT)]
Max = [[0] * MAX_RES for _ in range(MAX_DEPT)]
Need = [[0] * MAX_RES for _ in range(MAX_DEPT)]
Avail = [0] * MAX_RES
finished = [False] * MAX_DEPT
safeSeq = [0] * MAX_DEPT
nDept = 0

def reset():
    global Total, Alloc, Max, Need, Avail, finished, safeSeq
    Total = [0] * MAX_RES
    Alloc = [[0] * MAX_RES for _ in range(MAX_DEPT)]
    Max = [[0] * MAX_RES for _ in range(MAX_DEPT)]
    Need = [[0] * MAX_RES for _ in range(MAX_DEPT)]
    Avail = [0] * MAX_RES
    finished = [False] * MAX_DEPT
    safeSeq = [0] * MAX_DEPT

def computeNeed():
    for i in range(nDept):
        for j in range(MAX_RES):
            Need[i][j] = Max[i][j] - Alloc[i][j]

def computeAvail():
    sumAlloc = [0] * MAX_RES
    for i in range(nDept):
        for j in range(MAX_RES):
            sumAlloc[j] += Alloc[i][j]
    for j in range(MAX_RES):
        Avail[j] = Total[j] - sumAlloc[j]

def isSafe():
    work = Avail[:]
    for i in range(nDept):
        finished[i] = False

    count = 0
    while count < nDept:
        found = False
        for i in range(nDept):
            if not finished[i]:
                if all(Need[i][j] <= work[j] for j in range(MAX_RES)):
                    for j in range(MAX_RES):
                        work[j] += Alloc[i][j]
                    safeSeq[count] = i
                    count += 1
                    finished[i] = True
                    found = True
                    break
        if not found:
            print("System is NOT in safe state.")
            return False

    print("System is in safe state.")
    print("Safe sequence:", " ".join(f"D{safeSeq[i]}" for i in range(nDept)))
    return True

def requestResource(dept, reqPrint, reqScan):
    req = [reqPrint, reqScan]

    if any(req[j] > Need[dept][j] for j in range(MAX_RES)):
        print("Request denied: exceeds max need.")
        return False

    if any(req[j] > Avail[j] for j in range(MAX_RES)):
        print("Request denied: not enough available resources.")
        return False

    savedAlloc = copy.deepcopy(Alloc)
    savedAvail = Avail[:]

    for j in range(MAX_RES):
        Alloc[dept][j] += req[j]
        Avail[j] -= req[j]

    computeNeed()

    if isSafe():
        print(f"Request GRANTED to D{dept}")
        return True
    else:
        # rollback
        for i in range(nDept):
            for j in range(MAX_RES):
                Alloc[i][j] = savedAlloc[i][j]
        for j in range(MAX_RES):
            Avail[j] = savedAvail[j]
        computeNeed()
        print("Request DENIED (unsafe)")
        return False


def run_test_case(case_no):
    global nDept
    reset()

    print(f"\n========== TEST CASE {case_no} ==========")

    print(f"Enter number of departments (max {MAX_DEPT}): ", end="")
    nDept = int(input())

    print("Enter total printers and scanners: ", end="")
    Total[0], Total[1] = map(int, input().split())

    print("Enter allocation matrix:")
    for i in range(nDept):
        print(f"D{i}: ", end="")
        Alloc[i][0], Alloc[i][1] = map(int, input().split())

    print("Enter max matrix:")
    for i in range(nDept):
        print(f"D{i}: ", end="")
        Max[i][0], Max[i][1] = map(int, input().split())

    computeNeed()
    computeAvail()

    print("\nInitial State:")
    print(f"Available: printers={Avail[0]}, scanners={Avail[1]}")

    isSafe()

    print(f"\nEnter request (dept printers scanners): ", end="")
    dept, p, s = map(int, input().split())

    if 0 <= dept < nDept:
        requestResource(dept, p, s)
    else:
        print("Invalid department")

# 🔥 Run TWO test cases
run_test_case(1)
run_test_case(2)



========== TEST CASE 1 ==========
Enter number of departments (max 5): 4
Enter total printers and scanners: 3 2
Enter allocation matrix:
D0: 1 0
D1: 1 1
D2: 0 1
D3: 1 0
Enter max matrix:
D0: 2 1
D1: 2 2
D2: 1 2
D3: 2 1

Initial State:
Available: printers=0, scanners=0
System is NOT in safe state.

Enter request (dept printers scanners): 1 1 1
Request denied: not enough available resources.

========== TEST CASE 2 ==========
Enter number of departments (max 5): 3 
Enter total printers and scanners: 3 3
Enter allocation matrix:
D0: 0 1
D1: 2 0
D2: 0 0
Enter max matrix:
D0: 2 2
D1: 2 2
D2: 1 2

Initial State:
Available: printers=1, scanners=2
System is in safe state.
Safe sequence: D1 D0 D2

Enter request (dept printers scanners): 1 0 0
System is in safe state.
Safe sequence: D1 D0 D2
Request GRANTED to D1
